# X03 RDD2022 pilot: train
Private research run. Japan is not used.

In [ ]:
from pathlib import Path
Path('metrics.py').write_text('"""Explicit operational matching/geometry; AP is computed by the pinned YOLO validator."""\nimport numpy as np\n\ndef grid_cell(x,y,width,height):\n    gain=640/max(width,height)\n    # Ultralytics LetterBox rounds resized dimensions then splits integer padding.\n    left=round((640-round(width*gain))/2-.1)\n    top=round((640-round(height*gain))/2-.1)\n    return min(39,max(0,int((y*gain+top)/16))),min(39,max(0,int((x*gain+left)/16)))\n\ndef xyxy(boxes, width, height):\n    out=[]\n    for c,x,y,w,h in boxes:\n        out.append([c,(x-w/2)*width,(y-h/2)*height,(x+w/2)*width,(y+h/2)*height])\n    return np.asarray(out,dtype=float).reshape(-1,5)\n\ndef iou(a,b):\n    lo=np.maximum(a[:2],b[:,:2]); hi=np.minimum(a[2:],b[:,2:])\n    inter=np.maximum(0,hi-lo).prod(1)\n    area=np.maximum(0,a[2:]-a[:2]).prod()\n    return inter/(area+np.maximum(0,b[:,2:]-b[:,:2]).prod(1)-inter+1e-12)\n\ndef match(gt,pred,threshold,overlap=.5):\n    # GT: class,x1,y1,x2,y2; prediction: x1,y1,x2,y2,confidence,class.\n    used=set(); pairs=[]; fp=[]\n    for i in sorted(range(len(pred)),key=lambda i:-pred[i,4]):\n        if pred[i,4]<threshold: continue\n        eligible=[j for j in range(len(gt)) if j not in used and gt[j,0]==pred[i,5]]\n        if eligible:\n            overlaps=iou(pred[i,:4],gt[eligible,1:]); j=int(np.argmax(overlaps))\n            if overlaps[j]>=overlap:\n                used.add(eligible[j]); pairs.append((i,eligible[j])); continue\n        fp.append(i)\n    return pairs,fp,[j for j in range(len(gt)) if j not in used]\n\ndef masks(gt,width,height):\n    # Canonicalise sub-micropixel cancellation from xywh -> xyxy -> side lengths.\n    # Exact 16px/32px boundaries must not drift below strict cutoffs.\n    sides=np.round((gt[:,3:5]-gt[:,1:3])*640/max(width,height),6)\n    small=sides.prod(1)<32**2; thin=sides.min(1)<16\n    return dict(all=np.ones(len(gt),dtype=bool),small=small,thin=thin,small_or_thin=small|thin,\n                B_eligible=(small|thin)&(sides.min(1)>=8))\n\ndef operating(records,predictions,threshold):\n    totals={k:dict(tp=0,gt=0) for k in [\'all\',\'small\',\'thin\',\'small_or_thin\',\'B_eligible\']}\n    fps=0; damage_images=0; images_with_miss=0; per_image=[]\n    for r in records:\n        gt=xyxy(r[\'boxes\'],r[\'width\'],r[\'height\']); pred=np.asarray(predictions[r[\'id\']],float).reshape(-1,6)\n        pairs,fp,miss=match(gt,pred,threshold); mm=masks(gt,r[\'width\'],r[\'height\']); hit={j for _,j in pairs}\n        for k,m in mm.items():\n            totals[k][\'gt\']+=int(m.sum()); totals[k][\'tp\']+=sum(bool(m[j]) for j in hit)\n        fps+=len(fp); damage_images+=bool(len(gt)); images_with_miss+=bool(miss)\n        per_image.append(dict(id=r[\'id\'],group=r[\'group\'],tp=len(pairs),fp=len(fp),fn=len(miss),\n                              target_gt=int(mm[\'small_or_thin\'].sum()),target_tp=sum(bool(mm[\'small_or_thin\'][j]) for j in hit)))\n    for v in totals.values(): v[\'recall\']=v[\'tp\']/v[\'gt\'] if v[\'gt\'] else None; v[\'sparse\']=v[\'gt\']<30\n    tp=totals[\'all\'][\'tp\']; fn=totals[\'all\'][\'gt\']-tp\n    target_fn=totals[\'small_or_thin\'][\'gt\']-totals[\'small_or_thin\'][\'tp\']\n    return dict(threshold=threshold,geometry=totals,precision=tp/(tp+fps) if tp+fps else 0,\n                fp_per_image=fps/max(1,len(records)),misses_per_image=fn/max(1,len(records)),\n                small_or_thin_share_of_misses=target_fn/fn if fn else None,\n                small_or_thin_share_of_labels=totals[\'small_or_thin\'][\'gt\']/totals[\'all\'][\'gt\'] if totals[\'all\'][\'gt\'] else None,\n                damage_images_with_miss_fraction=images_with_miss/damage_images if damage_images else None,\n                images=len(records),per_image=per_image)\n\ndef select_threshold(records,predictions):\n    # Fixed reproducible grid; predictions below .001 are not retained.\n    rows=[]\n    for t in np.linspace(.001,.999,200):\n        m=operating(records,predictions,float(t)); p=m[\'precision\']; rec=m[\'geometry\'][\'all\'][\'recall\'] or 0\n        rows.append((float(t),p,rec,2*p*rec/(p+rec) if p+rec else 0))\n    eligible=[r for r in rows if r[1]>=.7]\n    chosen=max(eligible,key=lambda r:(r[2],r[0])) if eligible else max(rows,key=lambda r:(r[3],r[0]))\n    return chosen[0],dict(rule=\'precision>=0.70, maximum recall, ties higher threshold\' if eligible else \'best F1 fallback\',grid=rows)\n\ndef average_precision_binary(labels,scores):\n    labels=np.asarray(labels,bool); scores=np.asarray(scores,float)\n    if not labels.any(): return None\n    # Tie-aware stepwise average precision (same definition as sklearn average_precision_score).\n    order=np.argsort(-scores,kind=\'stable\'); y=labels[order]; s=scores[order]\n    ends=np.r_[np.where(np.diff(s)!=0)[0],len(s)-1]\n    tp=np.cumsum(y)[ends]; recall=tp/y.sum(); precision=tp/(ends+1)\n    return float(np.sum(np.diff(np.r_[0,recall])*precision))\n\ndef bootstrap(records,per_image,n=1000):\n    groups={}\n    for x in per_image: groups.setdefault(x[\'group\'],[]).append(x)\n    values=list(groups.values()); rng=np.random.default_rng(42); rows=[]\n    for _ in range(n):\n        sample=[x for i in rng.integers(0,len(values),len(values)) for x in values[i]]\n        tp=sum(x[\'tp\'] for x in sample); fp=sum(x[\'fp\'] for x in sample); fn=sum(x[\'fn\'] for x in sample)\n        tg=sum(x[\'target_gt\'] for x in sample); tt=sum(x[\'target_tp\'] for x in sample)\n        rows.append([tp/max(1,tp+fn),tp/max(1,tp+fp),fp/len(sample),tt/tg if tg else np.nan])\n    return {k:np.nanpercentile(np.asarray(rows)[:,i],[2.5,97.5]).tolist() for i,k in enumerate([\'recall\',\'precision\',\'fp_per_image\',\'small_or_thin_recall\'])}\n')
Path('pilot.py').write_text('"""Kaggle smoke/recovery, stock training and source-only evaluation."""\nimport argparse\nimport collections\nimport hashlib\nimport gzip\nimport json\nimport os\nimport platform\nimport shutil\nimport subprocess\nimport sys\nimport tarfile\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nimport yaml\nfrom PIL import Image, ImageDraw\nfrom ultralytics import YOLO, __version__\nfrom metrics import xyxy,match,masks,operating,select_threshold,average_precision_binary,bootstrap,grid_cell\n\nROOT=Path(\'/kaggle/working\'); OUT=ROOT/\'results\'; OUT.mkdir(exist_ok=True)\n\ndef save(name,obj):\n    body=json.dumps(obj,indent=2,default=str)\n    if name.startswith(\'predictions-\'):\n        # Low-threshold full-country predictions can be large; compress transfer artifacts.\n        with gzip.open(OUT/(name+\'.gz\'),\'wt\',encoding=\'utf-8\',compresslevel=3) as f: f.write(body)\n    else: (OUT/name).write_text(body)\n\ndef setup():\n    assert __version__==\'8.4.192\', __version__\n    assert torch.cuda.is_available(), \'GPU required for this stage\'\n    if not (ROOT/\'prepared/images\').exists():\n        archive=next(Path(\'/kaggle/input\').rglob(\'prepared.tar.gz\'))\n        with tarfile.open(archive) as f: f.extractall(ROOT,filter=\'data\')\n    env=dict(ultralytics=__version__,python=platform.python_version(),torch=torch.__version__,\n             cuda=torch.version.cuda,gpu=torch.cuda.get_device_name(0),vram_bytes=torch.cuda.get_device_properties(0).total_memory)\n    save(\'environment.json\',env)\n    (OUT/\'pip-freeze.txt\').write_text(subprocess.check_output([sys.executable,\'-m\',\'pip\',\'freeze\'],text=True))\n    return json.loads((ROOT/\'prepared/manifest.json\').read_text())\n\ndef data_yaml(train=\'train\',val=\'selection\',suffix=\'\'):\n    p=ROOT/f\'data{suffix}.yaml\'\n    p.write_text(yaml.safe_dump(dict(path=str(ROOT/\'prepared\'),train=str(ROOT/\'prepared\'/f\'{train}.txt\'),\n                                   val=str(ROOT/\'prepared\'/f\'{val}.txt\'),names=[\'D00\',\'D10\',\'D20\',\'D40\'])))\n    return str(p)\n\ndef settings(batch,epochs=50):\n    return dict(data=data_yaml(),epochs=epochs,imgsz=640,batch=batch,device=0,amp=True,seed=42,\n                deterministic=True,workers=2,cache=False,save=True,save_period=5,patience=100,\n                project=str(ROOT/\'training\'),exist_ok=True,plots=True)\n\ndef smoke(records):\n    ids=[r[\'id\'] for r in records if r[\'split\']==\'train\']\n    # Deterministic small execution test, not the scientific pilot dataset.\n    ids=sorted(ids,key=lambda s:hashlib.sha256(s.encode()).hexdigest())[:128]\n    (ROOT/\'prepared/smoke.txt\').write_text(\'\\n\'.join(str(ROOT/\'prepared/images\'/f\'{i}.jpg\') for i in ids))\n    vals=[r[\'id\'] for r in records if r[\'split\']==\'selection\'][:64]\n    (ROOT/\'prepared/smokeval.txt\').write_text(\'\\n\'.join(str(ROOT/\'prepared/images\'/f\'{i}.jpg\') for i in vals))\n    smoke_data=data_yaml(\'smoke\',\'smokeval\',\'-smoke\')\n    class PlannedInterruption(Exception): pass\n    def interrupt(trainer):\n        if trainer.epoch==0: raise PlannedInterruption(\'Deliberate interruption after epoch checkpoint\')\n    attempts=[]\n    for modelname in [\'yolo11s.pt\',\'yolo11n.pt\']:\n        for batch in [16,8,4]:\n            model=YOLO(modelname); model.add_callback(\'on_model_save\',interrupt)\n            args=settings(batch,2); args.update(data=smoke_data,name=\'smoke\')\n            try:\n                model.train(**args)\n                raise RuntimeError(\'interruption callback did not run\')\n            except PlannedInterruption: pass\n            except torch.cuda.OutOfMemoryError:\n                attempts.append(dict(model=modelname,batch=batch,status=\'OOM\')); del model; torch.cuda.empty_cache(); continue\n            checkpoint=ROOT/\'training/smoke/weights/last.pt\'\n            state=torch.load(checkpoint,map_location=\'cpu\',weights_only=False)\n            assert state[\'epoch\']==0 and state[\'optimizer\'] is not None\n            resume_model=YOLO(checkpoint)\n            seen=[]\n            resume_model.add_callback(\'on_train_start\',lambda t:seen.append(dict(start_epoch=t.start_epoch,optimizer_state_entries=len(t.optimizer.state))))\n            resume_model.train(resume=True,device=0)\n            assert seen[0][\'start_epoch\']==1 and seen[0][\'optimizer_state_entries\']>0,seen\n            save(\'smoke.json\',dict(status=\'passed\',model=modelname,batch=batch,recovery=seen,attempts=attempts))\n            shutil.copy2(ROOT/\'training/smoke/results.csv\',OUT/\'smoke-training.csv\')\n            return\n    raise RuntimeError(\'Both model sizes failed at all batches\')\n\ndef assignment_diagnostics(model,records):\n    from ultralytics.utils.tal import TaskAlignedAssigner\n    original=TaskAlignedAssigner.forward; rows=[]\n    def wrapped(self,pd_scores,pd_bboxes,anc_points,gt_labels,gt_bboxes,mask_gt,**extra):\n        original_gt=gt_bboxes.detach().clone()\n        result=original(self,pd_scores,pd_bboxes,anc_points,gt_labels,gt_bboxes,mask_gt,**extra)\n        _,_,scores,fg,idx=result\n        for b in range(len(gt_labels)):\n            for j in range(gt_labels.shape[1]):\n                if not mask_gt[b,j].item(): continue\n                sel=fg[b]&(idx[b]==j); size=(original_gt[b,j,2:]-original_gt[b,j,:2]).detach().cpu().tolist()\n                rows.append(dict(cls=int(gt_labels[b,j].item()),width=size[0],height=size[1],positives=int(sel.sum()),score_mass=float(scores[b,sel].sum())))\n        return result\n    TaskAlignedAssigner.forward=wrapped\n    net=model.model.to(\'cuda\'); net.eval()\n    try:\n        from ultralytics.data import build_yolo_dataset,build_dataloader\n        args=model.overrides.copy()\n        from ultralytics.cfg import get_cfg\n        if isinstance(net.args,dict): net.args=get_cfg(overrides=net.args)\n        cfg=get_cfg(overrides=dict(imgsz=640,task=\'detect\',mode=\'train\',augment=False))\n        train=[]\n        for country in [\'India\',\'Czech\']:\n            train.extend(sorted([r for r in records if r[\'split\']==\'train\' and r[\'country\']==country],key=lambda r:hashlib.sha256(r[\'id\'].encode()).hexdigest())[:128])\n        txt=ROOT/\'assignment.txt\'; txt.write_text(\'\\n\'.join(str(ROOT/\'prepared/images\'/f"{r[\'id\']}.jpg") for r in train))\n        ds=build_yolo_dataset(cfg,str(txt),8,dict(names={i:c for i,c in enumerate([\'D00\',\'D10\',\'D20\',\'D40\'])},nc=4,channels=3),mode=\'val\',rect=False,stride=32)\n        loader=build_dataloader(ds,8,0,shuffle=False)\n        with torch.no_grad():\n            for batch in loader:\n                batch={k:v.to(\'cuda\') if isinstance(v,torch.Tensor) else v for k,v in batch.items()}\n                batch[\'img\']=batch[\'img\'].float()/255\n                net.loss(batch)\n        summaries={}\n        for name,subset in [(\'all\',rows),(\'small_or_thin\',[r for r in rows if r[\'width\']*r[\'height\']<1024 or min(r[\'width\'],r[\'height\'])<16]),(\'other\',[r for r in rows if r[\'width\']*r[\'height\']>=1024 and min(r[\'width\'],r[\'height\'])>=16])]:\n            summaries[name]=dict(boxes=len(subset),zero_positive_fraction=sum(r[\'positives\']==0 for r in subset)/len(subset) if subset else None,\n                                  mean_positives=float(np.mean([r[\'positives\'] for r in subset])) if subset else None,\n                                  mean_score_mass=float(np.mean([r[\'score_mass\'] for r in subset])) if subset else None)\n        save(\'A-assignment.json\',dict(records=rows,summary=summaries,images=len(train),note=\'Post-training frozen-best checkpoint, float32 evaluation mode, no gradients or optimizer steps; training-sample supervision diagnostic, not a log of actual training-time assignments.\'))\n    finally: TaskAlignedAssigner.forward=original\n\ndef train(records):\n    smoke_file=next(Path(\'/kaggle/input\').rglob(\'smoke.json\'))\n    smoke_info=json.loads(smoke_file.read_text()); assert smoke_info[\'status\']==\'passed\'\n    model=YOLO(smoke_info[\'model\']); times=[]; started=time.time(); epoch_start=[0]\n    model.add_callback(\'on_train_epoch_start\',lambda t:epoch_start.__setitem__(0,time.time()))\n    def end(t):\n        times.append(dict(epoch=t.epoch+1,seconds=time.time()-epoch_start[0],allocated=torch.cuda.max_memory_allocated(),reserved=torch.cuda.max_memory_reserved()))\n        save(\'timing.json\',dict(epochs=times,elapsed_seconds=time.time()-started))\n    model.add_callback(\'on_fit_epoch_end\',end)\n    model.train(**settings(smoke_info[\'batch\']),name=\'baseline\')\n    save(\'training-complete.json\',dict(status=\'complete\',epochs=50,model=smoke_info[\'model\'],batch=smoke_info[\'batch\'],seconds=time.time()-started))\n    for name in [\'best.pt\',\'last.pt\']: shutil.copy2(ROOT/\'training/baseline/weights\'/name,OUT/name)\n    for name in [\'results.csv\',\'args.yaml\']: shutil.copy2(ROOT/\'training/baseline\'/name,OUT/name)\n    save(\'checkpoint-sha256.json\',{n:hashlib.sha256((OUT/n).read_bytes()).hexdigest() for n in [\'best.pt\',\'last.pt\']})\n\ndef predict(model,records,size):\n    data={}; paths=[str(ROOT/\'prepared/images\'/f"{r[\'id\']}.jpg") for r in records]\n    # A Python list is interpreted as in-memory PIL/numpy input by Ultralytics,\n    # which can eagerly load all images and use one huge batch. A text source is lazy.\n    input_list=ROOT/\'prediction-inputs.txt\'; input_list.write_text(\'\\n\'.join(paths))\n    for result in model.predict(source=str(input_list),imgsz=size,rect=False,conf=.001,iou=.7,max_det=300,batch=8,device=0,stream=True,verbose=False):\n        boxes=result.boxes\n        data[Path(result.path).stem]=torch.cat([boxes.xyxy,boxes.conf[:,None],boxes.cls[:,None]],1).cpu().tolist()\n    return data\n\ndef c_diagnostics(records,preds,threshold):\n    labels=[]; scores=[]; positives=0; collisions=0; positive_images=0\n    for r in records:\n        gt=xyxy(r[\'boxes\'],r[\'width\'],r[\'height\']); pred=np.asarray(preds[r[\'id\']],float).reshape(-1,6)\n        _,_,miss=match(gt,pred,threshold); m=masks(gt,r[\'width\'],r[\'height\'])[\'small_or_thin\']\n        target=np.zeros((40,40),bool); weak=np.zeros((40,40))\n        # Grid is on square letterboxed 640 view, not a stretched original image.\n        def cell(x,y): return grid_cell(x,y,r[\'width\'],r[\'height\'])\n        for j in miss:\n            if not m[j]: continue\n            y,x=cell((gt[j,1]+gt[j,3])/2,(gt[j,2]+gt[j,4])/2)\n            collisions+=int(target[y,x]); target[y,x]=True\n        for p in pred:\n            if .05<=p[4]<threshold:\n                y,x=cell((p[0]+p[2])/2,(p[1]+p[3])/2); weak[y,x]=max(weak[y,x],p[4])\n        positives+=int(target.sum()); positive_images+=int(target.any()); labels.extend(target.ravel()); scores.extend(weak.ravel())\n    prevalence=positives/max(1,len(labels)); ap=average_precision_binary(labels,scores)\n    return dict(positive_cells=positives,positive_images=positive_images,collisions=collisions,prevalence=prevalence,\n                weak_map_average_precision=ap,ap_over_prevalence=ap/prevalence if ap is not None and prevalence else None,\n                weak_map_empty_by_threshold=threshold<=.05,grid=\'40x40 square-letterbox coordinate system\')\n\ndef evaluate(records):\n    checkpoint=next(p for p in Path(\'/kaggle/input\').rglob(\'best.pt\') if p.parent.name==\'results\')\n    coco=list(Path(\'/kaggle/input\').rglob(\'yolo11s.pt\'))\n    save(\'initial-weight-provenance.json\',dict(files={str(p):hashlib.sha256(p.read_bytes()).hexdigest() for p in coco},\n         observed_training_download=\'https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo11s.pt\'))\n    model=YOLO(checkpoint); selected=[r for r in records if r[\'split\']==\'selection\']\n    # Keep diagnostics in the evaluation stage so library/API issues cannot spoil a completed training job.\n    assignment_diagnostics(YOLO(checkpoint),records)\n    assert all(r[\'country\'] in [\'India\',\'Czech\'] for r in selected)\n    selection_preds=predict(model,selected,640)\n    save(\'predictions-source-selection-640.json\',selection_preds)\n    threshold,decision=select_threshold(selected,selection_preds); save(\'threshold.json\',dict(value=threshold,selection_ids=[r[\'id\'] for r in selected],**decision))\n    summary={}; samples=[]\n    for country in [\'India\',\'Czech\',\'United_States\']:\n        subset=[r for r in records if r[\'country\']==country and r[\'split\'] in [\'source_audit\',\'unseen\']]\n        preds=predict(model,subset,640); half=predict(model,subset,320)\n        save(f\'predictions-{country}-640.json\',preds); save(f\'predictions-{country}-320.json\',half)\n        op=operating(subset,preds,threshold); op[\'bootstrap95\']=bootstrap(subset,op[\'per_image\']); op[\'C\']=c_diagnostics(subset,preds,threshold)\n        positive=[r for r in subset if r[\'boxes\']]; negative=[r for r in subset if not r[\'boxes\']]\n        op[\'positive_images_operating\']=operating(positive,preds,threshold) if positive else None\n        op[\'negative_images_operating\']=operating(negative,preds,threshold) if negative else None\n        op[\'scale320\']=operating(subset,half,threshold)\n        transitions={k:collections.Counter() for k in [\'all\',\'small_or_thin\',\'B_eligible\']}\n        for r in subset:\n            gt=xyxy(r[\'boxes\'],r[\'width\'],r[\'height\']); mm=masks(gt,r[\'width\'],r[\'height\'])\n            hits=[]\n            for predictions in [preds,half]:\n                pp=np.asarray(predictions[r[\'id\']],float).reshape(-1,6)\n                hits.append({j for _,j in match(gt,pp,threshold)[0]})\n            for k,counts in transitions.items():\n                for j in np.flatnonzero(mm[k]):\n                    state=\'both\' if j in hits[0] and j in hits[1] else \'640_only\' if j in hits[0] else \'320_only\' if j in hits[1] else \'neither\'\n                    counts[state]+=1\n        op[\'B_scale_transitions\']={k:dict(v) for k,v in transitions.items()}\n        byclass={}\n        for cls in range(4):\n            counts=dict(gt=0,tp=0,fp=0,small_or_thin_gt=0,small_or_thin_tp=0)\n            for r in subset:\n                gt=xyxy(r[\'boxes\'],r[\'width\'],r[\'height\']); pp=np.asarray(preds[r[\'id\']],float).reshape(-1,6)\n                pairs,fp,_=match(gt,pp,threshold)\n                counts[\'gt\']+=int((gt[:,0]==cls).sum()); counts[\'tp\']+=sum(gt[j,0]==cls for _,j in pairs); counts[\'fp\']+=sum(pp[i,5]==cls for i in fp)\n                gm=masks(gt,r[\'width\'],r[\'height\'])[\'small_or_thin\']\n                counts[\'small_or_thin_gt\']+=int(((gt[:,0]==cls)&gm).sum())\n                counts[\'small_or_thin_tp\']+=sum(gt[j,0]==cls and gm[j] for _,j in pairs)\n            byclass[cls]={k:int(v) for k,v in counts.items()}\n            byclass[cls].update(precision=float(counts[\'tp\']/(counts[\'tp\']+counts[\'fp\'])) if counts[\'tp\']+counts[\'fp\'] else None,\n                                recall=float(counts[\'tp\']/counts[\'gt\']) if counts[\'gt\'] else None,sparse=counts[\'gt\']<30,\n                                small_or_thin_recall=float(counts[\'small_or_thin_tp\']/counts[\'small_or_thin_gt\']) if counts[\'small_or_thin_gt\'] else None,\n                                small_or_thin_sparse=counts[\'small_or_thin_gt\']<30)\n        op[\'byclass\']=byclass\n        (ROOT/\'prepared\'/f\'{country}.txt\').write_text(\'\\n\'.join(str(ROOT/\'prepared/images\'/f"{r[\'id\']}.jpg") for r in subset))\n        val=model.val(data=data_yaml(val=country,suffix=\'-\'+country),imgsz=640,rect=False,batch=8,device=0,conf=.001,iou=.7,max_det=300,plots=False,verbose=False,project=str(ROOT/\'validation\'),name=country)\n        op[\'AP\']=dict(map50=float(val.box.map50),map50_95=float(val.box.map),per_class_ap=val.box.all_ap.tolist(),ap_class_index=val.box.ap_class_index.tolist())\n        summary[country]=op\n        # Fixed hash sample: no cherry-picking by apparent severity.\n        for r in sorted(subset,key=lambda r:hashlib.sha256(r[\'id\'].encode()).hexdigest())[:12]:\n            im=Image.open(ROOT/\'prepared/images\'/f"{r[\'id\']}.jpg").convert(\'RGB\'); draw=ImageDraw.Draw(im)\n            gt=xyxy(r[\'boxes\'],r[\'width\'],r[\'height\']); pp=np.asarray(preds[r[\'id\']],float).reshape(-1,6)\n            _,_,miss=match(gt,pp,threshold)\n            for j,b in enumerate(gt): draw.rectangle(b[1:].tolist(),outline=\'red\' if j in miss else \'green\',width=3)\n            for b in pp:\n                if b[4]>=threshold: draw.rectangle(b[:4].tolist(),outline=\'cyan\',width=2)\n            im.thumbnail((960,960)); im.save(OUT/f"example-{r[\'id\']}.jpg")\n    summary[\'source_macro_AP\']={k:float(np.mean([summary[c][\'AP\'][k] for c in [\'India\',\'Czech\']])) for k in [\'map50\',\'map50_95\']}\n    save(\'summary.json\',summary)\n    # Verify feature extraction without caching the dataset.\n    hooks=[]; shapes={}; probe=YOLO(checkpoint).model.to(\'cuda\').float().eval()\n    for idx in [16,19]:\n        hooks.append(probe.model[idx].register_forward_hook(lambda m,i,o,idx=idx:shapes.__setitem__(str(idx),list(o.shape))))\n    from ultralytics.data.augment import LetterBox\n    im=np.asarray(Image.open(ROOT/\'prepared/images\'/f"{selected[0][\'id\']}.jpg").convert(\'RGB\'))\n    im=LetterBox(new_shape=(640,640),auto=False)(image=im)\n    tensor=torch.from_numpy(np.ascontiguousarray(im.transpose(2,0,1))).to(\'cuda\').float()[None]/255\n    try:\n        with torch.no_grad(): probe(tensor)\n    finally:\n        for h in hooks: h.remove()\n    assert shapes[\'16\'][-2:]==[80,80] and shapes[\'19\'][-2:]==[40,40],shapes\n    summary[\'feature_shapes\']=shapes\n    save(\'summary.json\',summary)\n    save(\'storage.json\',dict(output_bytes=sum(p.stat().st_size for p in OUT.rglob(\'*\') if p.is_file())))\n    print(json.dumps({c:{k:v for k,v in m.items() if k in [\'AP\',\'precision\',\'fp_per_image\',\'C\']} for c,m in summary.items() if isinstance(m,dict)},indent=2),flush=True)\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser(); parser.add_argument(\'stage\',choices=[\'smoke\',\'train\',\'evaluate\']); args=parser.parse_args()\n    records=setup()\n    try: globals()[args.stage](records)\n    finally:\n        # Inputs remain in the immutable preparation artifact, including on diagnostic failures.\n        shutil.rmtree(ROOT/\'prepared\',ignore_errors=True)\n')


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.4.192'], check=True)
subprocess.run([sys.executable, '-u', 'pilot.py', 'train'], check=True)